# AgentOps Lab 09 - Optimize the trajectory

This notebook starts with a deliberately inefficient agent. It succeeds, but it wastes model calls, repeats searches, repeats log queries, and reflects after it already has enough evidence.

The optimization target is not "minimize tokens." The stronger target is: find the shortest reliable trajectory to a correct result.

## Inefficient trajectory

```mermaid
flowchart TD
    A["Plan"] --> B["search incidents"]
    B --> C["think"]
    C --> D["query health"]
    D --> E["think"]
    E --> F["search incidents again"]
    F --> G["retrieve runbook"]
    G --> H["think"]
    H --> I["query logs"]
    I --> J["reflection"]
    J --> K["query logs again"]
    K --> L["answer"]
```

It succeeds, but success alone is a blunt metric.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.trajectory_optimization import INEFFICIENT, OPTIMIZED, compare_profiles, efficiency_score, optimization_rules


## Compare before and after

The optimized path keeps the evidence needed for correctness and removes redundant thinking and tool calls.

In [ ]:
comparison = compare_profiles()
comparison


## Efficiency score

For this teaching lab, use a simple score:

```python
efficiency_score = success / (latency_seconds + cost_weight + trajectory_length)
```

The exact formula is less important than the habit: compare correct runs by latency, cost, and path length, not vibes.

In [ ]:
print("inefficient:", efficiency_score(INEFFICIENT))
print("optimized:", efficiency_score(OPTIMIZED))


## Optimization rules

These are the rules the learner applies to move from the wasteful trajectory to the shorter reliable one.

In [ ]:
for rule in optimization_rules():
    print("-", rule)


## What improved?

The optimized run keeps success while reducing model calls, tool calls, latency, cost, and trajectory length.

In [ ]:
comparison["improvement"]


## Exercises

- Remove `retrieve_runbook` from the optimized trajectory. Does recommendation support still pass?
- Add a cheaper but less reliable path. How do you decide whether to ship it?
- Add a trajectory cache for repeated `search_incidents` calls.
- Compare cost per successful task before and after optimization across ten tasks.

References: [Anthropic: Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), [Anthropic: demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).